### **Design**
- **`Config`**: Centralized configuration with properties for directory paths
- **`FontManager`**: Handles font loading and caching
- **`TextProcessor`**: Static methods for text processing operations
- **`ImageGenerator`**: Handles all image creation logic
- **`AudioProcessor`**: Manages TTS pipeline and audio processing
- **`VideoGenerator`**: Main orchestrator class
- **`TextSegment`**: dataclass to represent text with styling info

In [1]:
import os, shutil, subprocess
import re, json
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any, Tuple, Union
from contextlib import contextmanager
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
import multiprocessing as mp
from functools import partial

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import textwrap
from PIL import Image, ImageDraw, ImageFont
import soundfile as sf
from kokoro import KPipeline
import torch
import nltk

try:
    nltk.data.find("tokenizers/punkt_tab")
except nltk.downloader.DownloadError:
    nltk.download("punkt_tab", quiet=True)

from nltk.tokenize import sent_tokenize

In [2]:
# Configuration
@dataclass
class Config:
    screen_size: Tuple[int, int] = (720, 1280)
    font_size: int = 45
    min_font_size: int = 24
    padding: int = 80
    line_spacing: int = 10
    sample_rate: int = 24000
    voice: str = "af_heart"
    speed: float = 1.0
    image_display_duration: float = 5.0  # seconds for image display
    max_workers: int = min(8, mp.cpu_count())  # Max parallel processes
    ffmpeg_threads: int = 4
    
    # Directories
    base_dir: Path = Path("video-resource")
    
    @property
    def frame_dir(self) -> Path:
        return self.base_dir / "frames"
    
    @property
    def audio_dir(self) -> Path:
        return self.base_dir / "audio"
    
    @property
    def output_dir(self) -> Path:
        return self.base_dir / "output"
    
    @property
    def font_dir(self) -> Path:
        return self.base_dir / "fonts"
    
    @property
    def fonts(self) -> Dict[str, str]:
        return {
            "bold": str(self.font_dir / "BearSansUI-Bold.otf"),
            "italic": str(self.font_dir / "BearSansUI-Italic.otf"),
            "regular": str(self.font_dir / "BearSansUI-Regular.otf")
        }

In [3]:
@dataclass
class TextSegment:
    """Represents a segment of text with styling information."""
    text: str
    header: Optional[str] = None
    subtitle: Optional[str] = None
    is_quote: bool = False
    is_summary: bool = False
    is_image: bool = False
    image_path: Optional[str] = None

In [4]:
@dataclass 
class ProcessingTask:
    """Task for parallel processing."""
    segment: TextSegment
    frame_index: int
    config: Config

In [5]:
class FontManager:
    """Manages font loading and caching."""
    
    def __init__(self, config: Config):
        self.config = config
        self.cache: Dict[str, Dict[int, ImageFont.FreeTypeFont]] = {}
        self._load_fonts()
    
    def _load_fonts(self):
        """Pre-load and cache fonts in different sizes."""
        for style, path in self.config.fonts.items():
            self.cache[style] = {}
            for size in range(self.config.min_font_size, self.config.font_size + 11):
                try:
                    self.cache[style][size] = ImageFont.truetype(path, size)
                except OSError:
                    self.cache[style][size] = ImageFont.load_default()
    
    def get_font(self, style: str, size: int) -> ImageFont.FreeTypeFont:
        """Get a font with fallback to default."""
        return self.cache.get(style, {}).get(size, ImageFont.load_default())

In [6]:
class TextProcessor:
    """Handles text processing and chunking operations."""
    
    @staticmethod
    def split_text_with_images(text: str) -> List[Union[str, Dict[str, str]]]:
        """Split text into sentences and extract image references."""
        if not text:
            return []
        
        # Pattern to match <img path=...> tags
        img_pattern = re.compile(r'<img\s+path\s*=\s*([^>]+)>')
        
        parts = []
        current_pos = 0
        
        for match in img_pattern.finditer(text):
            # Add text before the image
            before_text = text[current_pos:match.start()].strip()
            if before_text:
                parts.extend(TextProcessor.split_text(before_text))
            
            # Add image reference
            image_path = match.group(1).strip().strip('"\'')
            parts.append({"type": "image", "path": image_path})
            
            current_pos = match.end()
        
        # Add remaining text
        remaining_text = text[current_pos:].strip()
        if remaining_text:
            parts.extend(TextProcessor.split_text(remaining_text))
        
        return parts
    
    @staticmethod
    def split_text(text: str) -> List[str]:
        """Split text into sentences, preserving numbered lists."""
        if not text:
            return []
        
        numbered_list_pattern = re.compile(r"^\d+\.\s+")
        chunks = []
        
        for line in text.strip().split('\n'):
            line = line.strip()
            if not line:
                continue
            if numbered_list_pattern.match(line):
                chunks.append(line)
            else:
                chunks.extend(sent_tokenize(line))
        
        return [s.strip() for s in chunks if s.strip()]
    
    @staticmethod
    def handle_colon_splits(text: str) -> Tuple[str, str]:
        """Split text at colon if present, return (before_colon, after_colon)."""
        if ':' in text:
            parts = text.split(':', 1)
            before = parts[0].strip() + ':'
            after = parts[1].strip() if len(parts) > 1 else ''
            return before, after
        return text, ''
    
    @staticmethod
    def chunk_long_text(text: str, threshold: int = 220, max_length: int = 200) -> List[str]:
        """Break long text into manageable chunks with ellipsis."""
        if len(text) <= threshold:
            return [text]

        chunks = textwrap.wrap(text, width=max_length, break_long_words=False, break_on_hyphens=False)
        
        if len(chunks) <= 1:
            return chunks

        modified_chunks = []
        for i, chunk in enumerate(chunks):
            if i > 0:
                chunk = "... " + chunk
            if i < len(chunks) - 1:
                chunk = chunk + "..."
            modified_chunks.append(chunk)

        return modified_chunks
    
    @staticmethod
    def wrap_text_by_pixels(draw: ImageDraw.Draw, text: str, font: ImageFont.FreeTypeFont, max_width: int) -> List[str]:
        """Wrap text to fit within pixel width."""
        lines = []
        words = text.split()
        if not words:
            return []
        
        current_line = words[0]
        for word in words[1:]:
            test_line = current_line + " " + word
            if draw.textlength(test_line, font=font) <= max_width:
                current_line = test_line
            else:
                lines.append(current_line)
                current_line = word
        lines.append(current_line)
        return lines

In [7]:
class ImageGenerator:
    """Handles image generation for video frames."""
    
    def __init__(self, config: Config, font_manager: FontManager):
        self.config = config
        self.font_manager = font_manager
    
    def create_text_image(self, segment: TextSegment) -> Image.Image:
        """Create an image from a text segment."""
        if segment.is_image and segment.image_path:
            return self._create_image_frame(segment.image_path)
        
        img = Image.new("RGB", self.config.screen_size, "white")
        draw = ImageDraw.Draw(img)
        
        max_width = self.config.screen_size[0] - 2 * self.config.padding
        y_pos = self.config.padding
        
        # Draw header
        if segment.header:
            y_pos = self._draw_header(draw, segment.header, max_width, y_pos)
        
        # Draw subtitle
        if segment.subtitle:
            y_pos = self._draw_subtitle(draw, segment.subtitle, max_width, y_pos)
        
        # Draw body
        if segment.text:
            self._draw_body(draw, segment, max_width, y_pos)
        
        return img
    
    def _create_image_frame(self, image_path: str) -> Image.Image:
        """Create a frame with the referenced image centered and fitted."""
        try:
            # Load the image
            source_img = Image.open(image_path)
            
            # Convert to RGB if necessary
            if source_img.mode != 'RGB':
                source_img = source_img.convert('RGB')
            
            # Calculate scaling to fit within screen while maintaining aspect ratio
            screen_w, screen_h = self.config.screen_size
            img_w, img_h = source_img.size
            
            # Calculate scale factor (fit within screen with padding)
            padding_factor = 0.9  # Use 90% of screen to leave some padding
            scale_w = (screen_w * padding_factor) / img_w
            scale_h = (screen_h * padding_factor) / img_h
            scale = min(scale_w, scale_h)
            
            # Resize image
            new_w = int(img_w * scale)
            new_h = int(img_h * scale)
            resized_img = source_img.resize((new_w, new_h), Image.Resampling.LANCZOS)
            
            # Create white background
            result = Image.new("RGB", self.config.screen_size, "white")
            
            # Calculate center position
            x_offset = (screen_w - new_w) // 2
            y_offset = (screen_h - new_h) // 2
            
            # Paste the resized image
            result.paste(resized_img, (x_offset, y_offset))
            
            return result
            
        except Exception as e:
            print(f"Warning: Could not load image {image_path}: {e}")
            # Return a placeholder frame with error message
            img = Image.new("RGB", self.config.screen_size, "white")
            draw = ImageDraw.Draw(img)
            font = self.font_manager.get_font("regular", self.config.font_size)
            
            error_text = f"Image not found:\n{image_path}"
            lines = error_text.split('\n')
            
            total_height = len(lines) * (font.getbbox("A")[3] + self.config.line_spacing)
            start_y = (self.config.screen_size[1] - total_height) // 2
            
            for i, line in enumerate(lines):
                text_width = draw.textlength(line, font=font)
                x = (self.config.screen_size[0] - text_width) // 2
                y = start_y + i * (font.getbbox("A")[3] + self.config.line_spacing)
                draw.text((x, y), line, font=font, fill="red")
            
            return img
    
    def _draw_header(self, draw: ImageDraw.Draw, header: str, max_width: int, y_pos: int) -> int:
        """Draw header text and return new y position."""
        font = self.font_manager.get_font("bold", int(self.config.font_size * 1.2))
        for line in TextProcessor.wrap_text_by_pixels(draw, header, font, max_width):
            draw.text((self.config.padding, y_pos), line, font=font, fill="black")
            y_pos += font.getbbox(line)[3] + self.config.line_spacing
        return y_pos + self.config.line_spacing
    
    def _draw_subtitle(self, draw: ImageDraw.Draw, subtitle: str, max_width: int, y_pos: int) -> int:
        """Draw subtitle text and return new y position."""
        font = self.font_manager.get_font("italic", self.config.font_size)
        for line in TextProcessor.wrap_text_by_pixels(draw, subtitle, font, max_width):
            draw.text((self.config.padding, y_pos), line, font=font, fill="gray")
            y_pos += font.getbbox(line)[3] + self.config.line_spacing
        return y_pos + self.config.line_spacing
    
    def _draw_body(self, draw: ImageDraw.Draw, segment: TextSegment, max_width: int, y_pos: int):
        """Draw body text with appropriate styling."""
        available_height = self.config.screen_size[1] - y_pos - self.config.padding
        
        # Determine font style and color
        font_style = "italic" if segment.is_quote or segment.is_summary else "regular"
        fill_color = "gray" if segment.is_summary else "black"
        
        # Find optimal font size
        font_size = self._find_optimal_font_size(draw, segment.text, font_style, max_width, available_height)
        font = self.font_manager.get_font(font_style, font_size)
        
        # Wrap text and calculate positioning
        lines = TextProcessor.wrap_text_by_pixels(draw, segment.text, font, max_width)
        total_height = sum(font.getbbox(line)[3] + self.config.line_spacing for line in lines) - self.config.line_spacing
        start_y = y_pos + (available_height - total_height) / 2
        
        # Draw quote/summary bar
        if segment.is_quote or segment.is_summary:
            self._draw_quote_bar(draw, start_y, total_height)
        
        # Draw text lines
        current_y = start_y
        for line in lines:
            draw.text((self.config.padding, current_y), line, font=font, fill=fill_color)
            current_y += font.getbbox(line)[3] + self.config.line_spacing
    
    def _find_optimal_font_size(self, draw: ImageDraw.Draw, text: str, font_style: str, max_width: int, available_height: int) -> int:
        """Find the largest font size that fits the available space."""
        for size in range(self.config.font_size, self.config.min_font_size - 1, -2):
            font = self.font_manager.get_font(font_style, size)
            lines = TextProcessor.wrap_text_by_pixels(draw, text, font, max_width)
            total_height = sum(font.getbbox(line)[3] + self.config.line_spacing for line in lines) - self.config.line_spacing
            if total_height <= available_height:
                return size
        return self.config.min_font_size
    
    def _draw_quote_bar(self, draw: ImageDraw.Draw, start_y: float, height: float):
        """Draw a vertical bar for quotes and summaries."""
        bar_width = 4
        bar_padding = 20
        bar_x0 = self.config.padding - bar_padding - bar_width
        bar_x1 = self.config.padding - bar_padding
        draw.rectangle([(bar_x0, start_y), (bar_x1, start_y + height)], fill="lightgray")

In [8]:
class AudioProcessor:
    """Handles audio generation and processing."""
    
    def __init__(self, config: Config):
        self.config = config
        # Note: GPU environment variable is set in the main process
        self.pipeline = None
        self._init_pipeline()
    
    def _init_pipeline(self):
        """Initialize the TTS pipeline."""
        os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
        self.pipeline = KPipeline(lang_code="a", repo_id="hexgrad/Kokoro-82M")
    
    def generate_audio(self, text: str) -> Tuple[np.ndarray, str]:
        """Generate audio from text and return audio data with synthesized text."""
        if self.pipeline is None:
            self._init_pipeline()
            
        full_audio = []
        text_parts = []
        
        for gs, _, audio in self.pipeline(text, voice=self.config.voice, speed=self.config.speed):
            if audio is not None:
                full_audio.append(audio)
                text_parts.append(gs)
        
        if not full_audio:
            raise ValueError(f"No audio generated for text: '{text}'")
        
        combined_audio = torch.cat(full_audio).squeeze().cpu().numpy()
        synthesized_text = "".join(text_parts)
        
        return combined_audio, synthesized_text
    
    def generate_silence(self, duration: float) -> np.ndarray:
        """Generate silence for specified duration."""
        num_samples = int(duration * self.config.sample_rate)
        return np.zeros(num_samples, dtype=np.float32)
    
    def split_audio_by_chunks(self, audio_data: np.ndarray, synthesized_text: str, chunks: List[str]) -> List[np.ndarray]:
        """Split audio data according to text chunks."""
        if len(chunks) <= 1:
            return [audio_data]
        
        total_samples = len(audio_data)
        total_chars = len(synthesized_text)
        audio_segments = []
        current_sample = 0
        start_search_index = 0
        
        for chunk in chunks:
            clean_chunk = chunk.replace("...", "").strip()
            
            try:
                chunk_start = synthesized_text.index(clean_chunk, start_search_index)
                chunk_end = chunk_start + len(clean_chunk)
                start_search_index = chunk_end
            except ValueError:
                chunk_start = 0
                chunk_end = len(clean_chunk)
                total_chars = len(synthesized_text) if total_chars == 0 else total_chars
            
            chunk_ratio = (chunk_end - chunk_start) / total_chars
            num_samples = int(total_samples * chunk_ratio)
            
            segment = audio_data[current_sample:current_sample + num_samples]
            audio_segments.append(segment)
            current_sample += num_samples
        
        return audio_segments

In [9]:
def process_single_task(task: ProcessingTask) -> Tuple[int, str, str, bool]:
    """Process a single task (for multiprocessing)."""
    try:
        # Initialize processors for this worker
        config = task.config
        font_manager = FontManager(config)
        image_generator = ImageGenerator(config, font_manager)
        audio_processor = AudioProcessor(config)
        
        segment = task.segment
        frame_index = task.frame_index
        
        # Handle image segments
        if segment.is_image:
            # Generate image frame
            image = image_generator.create_text_image(segment)
            image_path = config.frame_dir / f"frame_{frame_index:04d}.png"
            image.save(image_path)
            
            # Generate silence for image display duration
            silence = audio_processor.generate_silence(config.image_display_duration)
            audio_path = config.audio_dir / f"part_{frame_index:04d}.wav"
            sf.write(str(audio_path), silence, config.sample_rate)
            
            return frame_index, str(image_path), str(audio_path), True
        
        # Handle text segments
        chunks = TextProcessor.chunk_long_text(segment.text)
        
        # Prepare text for speech
        speech_text = segment.text
        if segment.is_quote:
            speech_text = f"Start quote. {segment.text} End quote."
        
        try:
            audio_data, synthesized_text = audio_processor.generate_audio(speech_text)
        except ValueError as e:
            print(f"Warning: {e}")
            return frame_index, "", "", False
        
        if len(chunks) <= 1:
            # Single chunk - simple processing
            # Clean display text
            display_text = segment.text
            if segment.is_quote:
                display_text = display_text.replace("Start quote.", "").replace("End quote.", "").strip()
            
            # Skip redundant display text
            if not segment.is_quote and not segment.is_summary:
                if display_text == segment.header or display_text == segment.subtitle:
                    display_text = None
            
            display_segment = TextSegment(
                text=display_text,
                header=segment.header,
                subtitle=segment.subtitle,
                is_quote=segment.is_quote,
                is_summary=segment.is_summary
            )
            
            # Generate and save image
            image = image_generator.create_text_image(display_segment)
            image_path = config.frame_dir / f"frame_{frame_index:04d}.png"
            image.save(image_path)
            
            # Save audio
            audio_path = config.audio_dir / f"part_{frame_index:04d}.wav"
            sf.write(str(audio_path), audio_data, config.sample_rate)
            
            return frame_index, str(image_path), str(audio_path), True
        else:
            # Multiple chunks - split audio
            audio_segments = audio_processor.split_audio_by_chunks(audio_data, synthesized_text, chunks)
            
            results = []
            for i, (chunk, audio_segment) in enumerate(zip(chunks, audio_segments)):
                current_frame_index = frame_index + i
                
                chunk_segment = TextSegment(
                    text=chunk,
                    header=segment.header,
                    subtitle=segment.subtitle,
                    is_quote=segment.is_quote,
                    is_summary=segment.is_summary
                )
                
                image = image_generator.create_text_image(chunk_segment)
                image_path = config.frame_dir / f"frame_{current_frame_index:04d}.png"
                image.save(image_path)
                
                audio_path = config.audio_dir / f"part_{current_frame_index:04d}.wav"
                sf.write(str(audio_path), audio_segment, config.sample_rate)
                
                results.append((current_frame_index, str(image_path), str(audio_path), True))
            
            return results
            
    except Exception as e:
        print(f"Error processing task {frame_index}: {e}")
        return frame_index, "", "", False

In [10]:
class VideoGenerator:
    """Main class that orchestrates the video generation process."""
    
    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()
        self.frame_index = 0
        
        self._setup_directories()
    
    def _setup_directories(self):
        """Create necessary directories."""
        for directory in [self.config.frame_dir, self.config.audio_dir, self.config.output_dir]:
            directory.mkdir(parents=True, exist_ok=True)
    
    @contextmanager
    def _cleanup_on_error(self):
        """Context manager to cleanup files on error."""
        try:
            yield
        except Exception as e:
            self._cleanup_temp_files()
            raise e
    
    def _cleanup_temp_files(self):
        """Remove temporary files."""
        for directory in [self.config.frame_dir, self.config.audio_dir]:
            if directory.exists():
                shutil.rmtree(directory)
                directory.mkdir(parents=True, exist_ok=True)
    
    def process_article(self, json_path: Path) -> str:
        """Process a JSON article and generate video."""
        with open(json_path, "r", encoding="utf-8") as f:
            article = json.load(f)
        
        title = article.get("title", "")
        subtitle = article.get("subtitle", "")
        sections = article.get("sections", [])
        
        with self._cleanup_on_error():
            # Collect all processing tasks
            tasks = []
            
            # Process title and subtitle
            tasks.extend(self._prepare_title_and_subtitle_tasks(title, subtitle))
            
            # Process sections
            for section in sections:
                tasks.extend(self._prepare_section_tasks(section, title))
            
            # Process tasks in parallel
            self._process_tasks_parallel(tasks)
            
            # Generate final video
            output_path = self._render_video(title)
            self._cleanup_temp_files()
            
            return str(output_path)
    
    def _prepare_title_and_subtitle_tasks(self, title: str, subtitle: str) -> List[ProcessingTask]:
        """Prepare tasks for title and subtitle processing."""
        tasks = []
        
        for sentence in TextProcessor.split_text(title):
            tasks.append(ProcessingTask(
                segment=TextSegment(text=sentence, header=sentence),
                frame_index=self.frame_index,
                config=self.config
            ))
            self.frame_index += 1
        
        for sentence in TextProcessor.split_text(subtitle):
            tasks.append(ProcessingTask(
                segment=TextSegment(text=sentence, header=title, subtitle=sentence),
                frame_index=self.frame_index,
                config=self.config
            ))
            self.frame_index += 1
        
        return tasks
    
    def _prepare_section_tasks(self, section: Dict[str, Any], main_title: str) -> List[ProcessingTask]:
        """Prepare tasks for a single section."""
        tasks = []
        section_title = section.get("title", "")
        summary_text = section.get("summary")
        header = section_title if section_title.lower() != main_title.lower() else main_title
        
        # Process section title
        if section_title and section_title.lower() != main_title.lower():
            for sentence in TextProcessor.split_text(section_title):
                tasks.append(ProcessingTask(
                    segment=TextSegment(text=sentence, header=section_title),
                    frame_index=self.frame_index,
                    config=self.config
                ))
                self.frame_index += 1
        
        # Process summary
        if summary_text:
            tasks.append(ProcessingTask(
                segment=TextSegment(text=f"Here is the summary of {section_title}:", header=header),
                frame_index=self.frame_index,
                config=self.config
            ))
            self.frame_index += 1
            
            for sentence in TextProcessor.split_text(summary_text):
                tasks.append(ProcessingTask(
                    segment=TextSegment(text=sentence, header=header, is_summary=True),
                    frame_index=self.frame_index,
                    config=self.config
                ))
                self.frame_index += 1
            
            tasks.append(ProcessingTask(
                segment=TextSegment(text="End of Summary. Now reading the main article:", header=header),
                frame_index=self.frame_index,
                config=self.config
            ))
            self.frame_index += 1
        
        # Process content with image support
        for para in section.get("content", []):
            is_quote = para.startswith("<start quote>")
            content = para.replace("<start quote>", "").replace("<end quote>", "").strip() if is_quote else para
            
            # Split content considering images
            parts = TextProcessor.split_text_with_images(content)
            
            for part in parts:
                if isinstance(part, dict) and part.get("type") == "image":
                    # Handle image
                    tasks.append(ProcessingTask(
                        segment=TextSegment(
                            text="Reference image:",
                            header=header,
                            is_image=False
                        ),
                        frame_index=self.frame_index,
                        config=self.config
                    ))
                    self.frame_index += 1
                    
                    # Add image frame
                    tasks.append(ProcessingTask(
                        segment=TextSegment(
                            text="",
                            header=header,
                            is_image=True,
                            image_path=part["path"]
                        ),
                        frame_index=self.frame_index,
                        config=self.config
                    ))
                    self.frame_index += 1
                    
                else:
                    # Handle text
                    sentence = part
                    
                    # Check for colon splits (for images that follow colons)
                    before_colon, after_colon = TextProcessor.handle_colon_splits(sentence)
                    
                    if after_colon:
                        # Split at colon
                        tasks.append(ProcessingTask(
                            segment=TextSegment(text=before_colon, header=header, is_quote=is_quote),
                            frame_index=self.frame_index,
                            config=self.config
                        ))
                        self.frame_index += 1
                        
                        # Continue with after colon part
                        sentence = after_colon
                    
                    tasks.append(ProcessingTask(
                        segment=TextSegment(text=sentence, header=header, is_quote=is_quote),
                        frame_index=self.frame_index,
                        config=self.config
                    ))
                    
                    # Account for potential chunking
                    chunks = TextProcessor.chunk_long_text(sentence)
                    self.frame_index += len(chunks)
        
        return tasks
    
    def _process_tasks_parallel(self, tasks: List[ProcessingTask]):
        """Process all tasks using multiprocessing."""
        print(f"Processing {len(tasks)} tasks with {self.config.max_workers} workers...")
        
        # Set GPU environment variable for all processes
        os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
        
        with ProcessPoolExecutor(max_workers=self.config.max_workers) as executor:
            # Submit all tasks
            future_to_task = {executor.submit(process_single_task, task): task for task in tasks}
            
            # Process completed tasks
            completed = 0
            for future in as_completed(future_to_task):
                try:
                    result = future.result()
                    completed += 1
                    if completed % 10 == 0:
                        print(f"Completed {completed}/{len(tasks)} tasks")
                except Exception as e:
                    task = future_to_task[future]
                    print(f"Task {task.frame_index} failed: {e}")
        
        print(f"✅ All {len(tasks)} tasks completed")
    
    def _render_video(self, title: str) -> Path:
        """Render final video using FFmpeg with multiple threads."""
        # Create filelist for FFmpeg
        filelist_path = self.config.frame_dir / "filelist.txt"
        with open(filelist_path, "w") as f:
            audio_files = sorted(self.config.audio_dir.glob("part_*.wav"))
            for audio_file in audio_files:
                idx = audio_file.stem.split("_")[1]
                image_file = self.config.frame_dir / f"frame_{idx}.png"
                
                if image_file.exists():
                    try:
                        duration = sf.info(str(audio_file)).duration
                        if duration >= 0.01:  # Skip very short audio clips
                            f.write(f"file '{image_file.resolve()}'\n")
                            f.write(f"duration {duration}\n")
                    except Exception:
                        continue
        
        # Combine all audio
        master_audio_path = self.config.audio_dir / "master_audio.wav"
        self._combine_audio_files(master_audio_path)
        
        # Run FFmpeg with multiple threads
        output_path = self.config.output_dir / f"{title}.mp4"
        command = [
            "ffmpeg",
            "-threads", str(self.config.ffmpeg_threads),  # Use multiple threads
            "-f", "concat", "-safe", "0", "-i", str(filelist_path),
            "-i", str(master_audio_path), 
            "-c:v", "libx264", 
            "-preset", "medium",  # Balance between speed and quality
            "-pix_fmt", "yuv420p",
            "-c:a", "copy", 
            "-shortest", 
            "-y", 
            str(output_path)
        ]
        
        try:
            print("🎬 Rendering video with FFmpeg...")
            result = subprocess.run(command, check=True, capture_output=True, text=True)
            print(f"✅ Video successfully generated: {output_path}")
            return output_path
        except subprocess.CalledProcessError as e:
            raise RuntimeError(f"FFmpeg rendering failed: {e.stderr}")
        except FileNotFoundError:
            raise RuntimeError("FFmpeg not found. Please install FFmpeg.")
    
    def _combine_audio_files(self, output_path: Path):
        """Combine all audio files into a master audio file."""
        audio_files = sorted(self.config.audio_dir.glob("part_*.wav"))
        if not audio_files:
            raise ValueError("No audio files found to combine")
        
        print(f"🎵 Combining {len(audio_files)} audio files...")
        combined_data = []
        for file_path in audio_files:
            data, sample_rate = sf.read(file_path)
            if sample_rate == self.config.sample_rate:
                combined_data.append(data)
        
        if combined_data:
            master_audio = np.concatenate(combined_data)
            sf.write(output_path, master_audio, self.config.sample_rate)
            print(f"✅ Master audio file created: {output_path}")

In [13]:
config = Config(
    max_workers=min(6, mp.cpu_count()),  # Use up to 6 cores
    ffmpeg_threads=4,
    image_display_duration=5.0,
    voice="af_heart"
)

generator = VideoGenerator(config)

print(f"Configuration:")
print(f"  - Max workers: {config.max_workers}")
print(f"  - FFmpeg threads: {config.ffmpeg_threads}")
print(f"  - Image display duration: {config.image_display_duration}s")
print(f"  - Screen size: {config.screen_size}")
print(f"  - Voice: {config.voice}")
print()

json_path = Path("video-resource/json-input/Money Stuff - A Drug-Trial Stock Sale.json")
print(f"📖 Processing article: {json_path}")
output_video = generator.process_article(json_path)

Configuration:
  - Max workers: 6
  - FFmpeg threads: 4
  - Image display duration: 5.0s
  - Screen size: (720, 1280)
  - Voice: af_heart

📖 Processing article: video-resource/json-input/Money Stuff - A Drug-Trial Stock Sale.json
Processing 107 tasks with 6 workers...
Task 0 failed: A process in the process pool was terminated abruptly while the future was running or pending.
Task 1 failed: A process in the process pool was terminated abruptly while the future was running or pending.
Task 2 failed: A process in the process pool was terminated abruptly while the future was running or pending.
Task 3 failed: A process in the process pool was terminated abruptly while the future was running or pending.
Task 4 failed: A process in the process pool was terminated abruptly while the future was running or pending.
Task 5 failed: A process in the process pool was terminated abruptly while the future was running or pending.
Task 6 failed: A process in the process pool was terminated abruptly wh

Process SpawnProcess-10:
Process SpawnProcess-9:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError:

ValueError: No audio files found to combine

In [ ]:
# Enhanced example usage with better error handling and progress tracking
def main():
    """Example usage of the VideoGenerator with enhanced features."""
    
    # Create config with custom settings
    config = Config(
        max_workers=min(6, mp.cpu_count()),  # Use up to 6 cores
        ffmpeg_threads=4,
        image_display_duration=5.0,
        voice="af_heart"
    )
    
    print(f"Configuration:")
    print(f"  - Max workers: {config.max_workers}")
    print(f"  - FFmpeg threads: {config.ffmpeg_threads}")
    print(f"  - Image display duration: {config.image_display_duration}s")
    print(f"  - Screen size: {config.screen_size}")
    print(f"  - Voice: {config.voice}")
    print()
    
    generator = VideoGenerator(config)
    
    # Process article
    json_path = Path("article.json")  # Replace with your JSON file path
    if json_path.exists():
        try:
            print(f"📖 Processing article: {json_path}")
            output_video = generator.process_article(json_path)
            print()
            print("=" * 50)
            print(f"🎉 SUCCESS! Video generated: {output_video}")
            print("=" * 50)
        except Exception as e:
            print(f"❌ Error generating video: {e}")
            import traceback
            traceback.print_exc()
    else:
        print(f"❌ Article file not found: {json_path}")
        print("\nExpected JSON format:")
        print("""
        {
            "title": "Your Article Title",
            "subtitle": "Optional subtitle",
            "sections": [
                {
                    "title": "Section Title",
                    "summary": "Optional section summary",
                    "content": [
                        "Regular paragraph text.",
                        "<start quote>This is a quoted text<end quote>",
                        "Text before image: <img path=path/to/image.jpg>",
                        "More text after image."
                    ]
                }
            ]
        }
        """)


# Additional utility functions for testing and validation
def validate_json_structure(json_path: Path) -> bool:
    """Validate that the JSON file has the expected structure."""
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        required_fields = ['title', 'sections']
        for field in required_fields:
            if field not in data:
                print(f"❌ Missing required field: {field}")
                return False
        
        if not isinstance(data['sections'], list):
            print("❌ 'sections' must be a list")
            return False
        
        for i, section in enumerate(data['sections']):
            if not isinstance(section, dict):
                print(f"❌ Section {i} must be a dictionary")
                return False
            
            if 'content' in section and not isinstance(section['content'], list):
                print(f"❌ Section {i} 'content' must be a list")
                return False
        
        print("✅ JSON structure is valid")
        return True
        
    except json.JSONDecodeError as e:
        print(f"❌ Invalid JSON format: {e}")
        return False
    except FileNotFoundError:
        print(f"❌ File not found: {json_path}")
        return False


def create_sample_json(output_path: Path = Path("sample_article.json")):
    """Create a sample JSON file for testing."""
    sample_data = {
        "title": "The Future of Artificial Intelligence",
        "subtitle": "Exploring the possibilities and challenges ahead",
        "sections": [
            {
                "title": "Introduction",
                "summary": "AI is rapidly transforming our world with both opportunities and challenges.",
                "content": [
                    "Artificial Intelligence has become one of the most transformative technologies of our time.",
                    "<start quote>AI will be the most important technology that humanity will ever develop.<end quote>",
                    "This article explores the current state and future possibilities: <img path=sample_images/ai_future.jpg>",
                    "We'll examine both the promising opportunities and the significant challenges that lie ahead."
                ]
            },
            {
                "title": "Current Applications",
                "content": [
                    "Today's AI applications span numerous industries and domains.",
                    "1. Healthcare diagnostics and treatment recommendations",
                    "2. Autonomous vehicles and transportation systems", 
                    "3. Natural language processing and communication",
                    "Here's an example of AI in healthcare: <img path=sample_images/ai_healthcare.jpg>",
                    "These applications demonstrate AI's growing influence on daily life."
                ]
            },
            {
                "title": "Future Prospects",
                "summary": "The future holds even more revolutionary AI developments across all sectors.",
                "content": [
                    "Looking ahead, AI promises even more revolutionary developments.",
                    "<start quote>The best way to predict the future is to invent it.<end quote>",
                    "Key areas of development include general AI, quantum computing integration, and human-AI collaboration.",
                    "The timeline for these advances: <img path=sample_images/ai_timeline.jpg>",
                    "However, we must also consider the ethical implications and ensure responsible development."
                ]
            }
        ]
    }
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(sample_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Sample JSON created: {output_path}")
    return output_path


if __name__ == "__main__":
    # Check if we should create a sample file
    import sys
    
    if len(sys.argv) > 1 and sys.argv[1] == "--create-sample":
        create_sample_json()
    elif len(sys.argv) > 1 and sys.argv[1] == "--validate":
        if len(sys.argv) > 2:
            validate_json_structure(Path(sys.argv[2]))
        else:
            print("Usage: python script.py --validate <json_file>")
    else:
        main()

In [ ]:
# Example usage
config = Config()
generator = VideoGenerator(config)

# Process article
json_path = Path(
    "video-resource/json-input/Money Stuff - A Drug-Trial Stock Sale.json"
)  # Replace with your JSON file path
output_video = generator.process_article(json_path)

Updates:
- Consider adding images from main article
- Incorporate "multiprocessing" to accelerate processing